# 10_5. Theta bound offline 분석 — trajectory 종합 비교

`10_1`~`10_4` 가 저장한 결과를 읽어 trajectory 간 차이를 본다.
**이 notebook 은 계산하지 않고 읽기만 한다.** 앞의 네 개를 먼저 실행해야 한다.

## 비교할 때 지켜야 할 것

trajectory 마다 cut 집합이 다르다. Gurobi 는 1,000 iteration, SA 는 163~240,
QA 는 19~25 이다. **같은 iteration 번호끼리 비교하면 안 된다.**

여기서 보는 것은 "각 trajectory 안에서 bound 전략이 어떤 효과를 내는가" 이고,
trajectory 간에는 **효과의 방향과 크기**를 비교한다.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from cflp_bd_qa_ex2.config.loader import load_config

cfg = load_config(ROOT / "config" / "experiments" / "pilot.yaml")
from cflp_bd_qa_ex2.evaluation.bound_analysis import (
    analysis_config_hash, select_analysis_results,
)

# 10_1~10_4와 동일한 분석 설정을 지정한다.
THETA_BITS_FIXED = 10
BIT_SWEEP = [4, 6, 8, 10, 12]
SNAPSHOT_ITERATIONS = [1, 5, 10, 25]
CUT_SLACK_RATIO = float(cfg["encoding"]["cut_slack"]["delta_ratio_to_theta"])
ANALYSIS_HASH = analysis_config_hash(
    cut_slack_ratio=CUT_SLACK_RATIO, reference_bits=THETA_BITS_FIXED,
    bit_sweep=BIT_SWEEP, snapshot_policy="last_executed_and_early",
    snapshot_iterations=SNAPSHOT_ITERATIONS,
)
OUT_DIR = ROOT / "results" / "theta_bound" / "analysis_v2" / ANALYSIS_HASH
SOURCES = ["gurobi", "sa", "pegasus", "zephyr"]
# 입력을 갱신했다면 비교할 trajectory_hash를 (source, instance_id)별로 지정한다.
TRAJECTORY_HASHES = {}

frames = []
for source in SOURCES:
    path = OUT_DIR / f"theta_bound_{source}.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path}가 없다. 해당 notebook을 먼저 실행하라")
    frames.append(pd.read_csv(path))
combined = select_analysis_results(
    pd.concat(frames, ignore_index=True), ANALYSIS_HASH,
    trajectory_hashes=TRAJECTORY_HASHES,
)
combined = combined[combined["snapshot"] == "last_executed"]
selected_inputs = combined[["trajectory_source", "instance_id", "trajectory_hash"]].drop_duplicates()
print("trajectory별 행 수:")
print(combined.groupby(["trajectory_source", "experiment"]).size().to_string())


## 1. trajectory 별 cut 수

bound 효과의 크기는 cut 수에 크게 의존한다. cut 하나당 slack bit 가
몇 개씩 붙기 때문이다.

In [ ]:
cut_counts = combined.groupby(["trajectory_source", "instance_id"])["num_cuts"].max().unstack()
display(cut_counts)


## 2. E1-A — 같은 bit 예산에서 bound 효과

θ bit 는 10으로 고정이므로 변수 변화는 **전부 cut slack bit 때문**이다.

In [ ]:
e1a = combined[combined.experiment == "E1-A"]

for metric in ("cut_slack_bits_total", "logical_variables"):
    table = e1a.pivot_table(index=["trajectory_source", "instance_id"],
                            columns="bound_strategy", values=metric)
    print(f"\n{metric}")
    display(table)


In [ ]:
# 정리 검증: S1 -> S2 -> S3 에서 cut slack bit 는 감소할 수 없다.
violations = []
for (source, tid), group in e1a.groupby(["trajectory_source", "instance_id"]):
    per_strategy = group.set_index("bound_strategy")["cut_slack_bits_total"]
    if not {"S1", "S2", "S3"} <= set(per_strategy.index):
        continue
    ok = per_strategy["S1"] <= per_strategy["S2"] <= per_strategy["S3"]
    if not ok:
        violations.append({"trajectory_source": source, "instance_id": tid,
                           "S1": per_strategy["S1"], "S2": per_strategy["S2"],
                           "S3": per_strategy["S3"]})

print(f"정리 위반: {len(violations)}건 (0 이어야 한다)")
if violations:
    display(pd.DataFrame(violations))


## 3. E1-B — 같은 목표 해상도에서 bit 절감

여기서는 θ bit 가 bound 에 따라 달라진다. **실제 δ_θ 는 목표값 이하**이므로
cut slack bit 증감은 단조적이지 않다.

In [ ]:
e1b = combined[combined.experiment == "E1-B"]
if e1b.empty:
    print("E1-B 결과가 없다")
else:
    display(e1b.pivot_table(index=["trajectory_source", "instance_id"],
                            columns="bound_strategy", values="theta_bits"))
    display(e1b.pivot_table(index=["trajectory_source", "instance_id"],
                            columns="bound_strategy", values="logical_variables"))


## 4. baseline 대비 변화

각 trajectory 안에서 S0(b=10) 을 기준으로 삼는다.
**trajectory 끼리 절대 변수 수를 비교하지 않는다** — cut 수가 다르기 때문이다.

In [ ]:
baseline = (e1a[e1a.bound_strategy == "S0"]
            .set_index(["trajectory_source", "instance_id"])["logical_variables"])

rows = []
for experiment, frame in (("E1-A", e1a), ("E1-B", e1b)):
    if frame.empty:
        continue
    work = frame.copy()
    work["baseline_variables"] = work.set_index(
        ["trajectory_source", "instance_id"]).index.map(baseline)
    work["change_pct"] = 100 * (work["logical_variables"] - work["baseline_variables"]) \
                         / work["baseline_variables"]
    summary = work.groupby(["trajectory_source", "bound_strategy"])["change_pct"].mean()
    rows.append(summary.rename(experiment))

if rows:
    change = pd.concat(rows, axis=1)
    print("S0(b=10) 대비 logical variable 변화율 (%)")
    display(change)


## 5. E2 — bit sweep

b 를 줄이면 θ 변수와 cut slack bit 가 **둘 다** 줄어든다.
해상도가 거칠어지는 대가이며, discretization 손실은 여기서 측정하지 않는다.

In [ ]:
e2 = combined[combined.experiment == "E2"]
if e2.empty:
    print("E2 결과가 없다")
else:
    for source in sorted(e2.trajectory_source.unique()):
        part = e2[e2.trajectory_source == source]
        table = part.pivot_table(index="bound_strategy", columns="theta_bits",
                                 values="logical_variables", aggfunc="mean")
        print(f"\n{source}: logical variables 평균 (행 bound, 열 theta bits)")
        display(table.round(1))


## 6. Offline Resource Trade-off

**자원과 정밀도만 본다.** feasibility, solution gap, embedding time 은
별도 실험이 필요하다. 여기서 "어떤 조합이 가장 좋다" 고 결론짓지 않는다.

In [ ]:
frontier = combined[combined.experiment.isin(["E1-A", "E1-B", "E2"])].copy()
# 정밀도 축은 **절대 delta_theta** 를 S0 기준으로 정규화해서 쓴다.
# theta_relative_resolution = delta_theta / theta_range = 1/(2^b - 1) 이라
# bound 전략 간 차이가 사라진다. 그 축을 쓰면 더 정밀한 전략이 불리해진다.
reference = (frontier[(frontier.experiment == "E1-A")
                      & (frontier.bound_strategy == "S0")]
             .set_index(["trajectory_source", "instance_id"])["delta_theta"])

frontier = frontier.copy()
frontier["delta_ref_s0"] = frontier.set_index(
    ["trajectory_source", "instance_id"]).index.map(reference)
frontier["normalized_delta_theta"] = frontier["delta_theta"] / frontier["delta_ref_s0"]

# instance 를 먼저 평균하지 않고 **instance 별로** Pareto 를 구한다.
pareto_input = frontier[[
    "trajectory_source", "instance_id", "bound_strategy", "theta_bits",
    "experiment", "logical_variables", "delta_theta", "normalized_delta_theta",
]].copy()

# 자원(변수 수)과 정밀도(상대 해상도)의 Pareto front.
# 변수는 적을수록, 상대 해상도는 작을수록 좋다.
def pareto_front(frame, resource="logical_variables", precision="normalized_delta_theta"):
    """두 지표 모두에서 지배당하지 않는 행만 남긴다 (둘 다 작을수록 좋다)."""
    keep = []
    for index, row in frame.iterrows():
        dominated = (
            (frame[resource] <= row[resource])
            & (frame[precision] <= row[precision])
            & ((frame[resource] < row[resource]) | (frame[precision] < row[precision]))
        ).any()
        if not dominated:
            keep.append(index)
    return frame.loc[keep]

for (source, tid), part in pareto_input.groupby(["trajectory_source", "instance_id"]):
    front = pareto_front(part).sort_values("logical_variables")
    print(f"\n{source} / {tid}: Pareto front (변수 수 vs S0 대비 delta_theta)")
    display(front[["bound_strategy", "theta_bits", "experiment",
                   "logical_variables", "normalized_delta_theta"]].round(6))


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

figures = OUT_DIR / "figures"
figures.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(pareto_input.trajectory_source.unique()),
                         figsize=(4.2 * pareto_input.trajectory_source.nunique(), 4.0),
                         squeeze=False)
for ax, source in zip(axes[0], sorted(pareto_input.trajectory_source.unique())):
    part = pareto_input[pareto_input.trajectory_source == source]
    for strategy, group in part.groupby("bound_strategy"):
        ax.scatter(group["logical_variables"], group["normalized_delta_theta"],
                   label=strategy, s=35, alpha=0.8)
    ax.set_yscale("log")
    ax.set_xlabel("Logical variables")
    ax.set_ylabel("delta_theta / delta_theta(S0)")
    ax.set_title(source)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)

fig.tight_layout()
path = figures / "fig_theta_bound_tradeoff.png"
fig.savefig(path, dpi=150)
plt.close(fig)
print(path)


## 7. 전 구간 비교 (iteration 별)

trajectory 마다 iteration 수가 다르므로 **같은 iteration 번호끼리 비교하지 않는다.**
각 trajectory 안에서 자원이 어떻게 늘어나는지만 본다.


In [ ]:
per_iteration_frames = []
for source in SOURCES:
    path = OUT_DIR / f"theta_bound_per_iteration_{source}.csv"
    if path.exists():
        per_iteration_frames.append(pd.read_csv(path))

if not per_iteration_frames:
    print("전 구간 결과가 없다")
else:
    per_iteration = select_analysis_results(
        pd.concat(per_iteration_frames, ignore_index=True), ANALYSIS_HASH,
        trajectory_hashes=TRAJECTORY_HASHES, expected_inputs=selected_inputs,
    )
    e1a_iter = per_iteration[per_iteration.experiment == "E1-A"]

    slopes = []
    for (source, tid, sid), group in e1a_iter.groupby(
        ["trajectory_source", "instance_id", "bound_strategy"]
    ):
        group = group.sort_values("benders_iteration")
        if len(group) < 2:
            continue
        span = group["benders_iteration"].iloc[-1] - group["benders_iteration"].iloc[0]
        growth = group["logical_variables"].iloc[-1] - group["logical_variables"].iloc[0]
        slopes.append({
            "trajectory_source": source, "instance_id": tid, "bound_strategy": sid,
            "iterations": int(span),
            "variables_added": int(growth),
            "variables_per_iteration": growth / span if span else float("nan"),
        })

    slope_frame = pd.DataFrame(slopes)
    print("iteration 당 logical variable 증가량")
    display(slope_frame.groupby(["trajectory_source", "bound_strategy"])
            ["variables_per_iteration"].mean().unstack().round(2))


## 해석

- **각 trajectory 안에서** bound 전략의 효과 방향이 같은지 본다. 다르면
  cut 구성(특히 `A_k < 0` 비율)이 trajectory 마다 다르다는 뜻이다.
- Gurobi trajectory 는 cut 이 실제 실행 MP의 활성 cut라 효과가 가장 크게 나타나지만,
  **QA 가 실제로 도달한 적 없는 상태**다. Primary 는 Pegasus/Zephyr 다.
- Pareto front 는 **자원과 정밀도** 두 축만 본다. 여기 있는 조합이
  실제로 좋은 해를 준다는 보장은 없다. SA/QA 검증이 따로 필요하다.
- 다음 단계에서 SA/QA 로 검증할 조합은 **사전에 정한 선별 규칙**으로
  고른다. 결과를 보고 고르면 사후 선택이 된다.
